In [1]:
import google.genai as genai
import config

/opt/anaconda3/envs/llm_env/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
client = genai.Client(api_key=config.api_key)

def generate_text(prompt):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    return response.text

# print(generate_text('capital of india'))

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=config.api_key
)

template = PromptTemplate(
    input_variables=["country"],
    template="What is the capital of {country}"
)

formatted_template = template.format(country="USA")


# response = llm.invoke(formatted_template)
# print(response.text)

In [4]:
chat_template = ChatPromptTemplate.from_messages([("system", "You are a indian history teacher who knows all the war happened"), 
                                                  ("human", "When {location} war happened?")])


formate_chat_template = chat_template.format_messages(location='Panipat')
# response = llm.invoke(formate_chat_template)
# print(response.text)

In [5]:
##Chain example

In [6]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableLambda

prompt1 = ChatPromptTemplate.from_messages([("system", "Your are a match teacher who provides math solutions."), 
                                                 ("human", "Add 2 and 2"), 
                                                 ("ai", "4"), 
                                                 ("human", "Add {num1} and {num2}")])

prompt2 = ChatPromptTemplate.from_messages([("system", "Your are a match teacher who provides math solutions. The output formate should be a json with the key answer"), 
                                                 ("human", "Multiply 4 to itself"), 
                                                 ("ai", """ {{ 
                                                     "answer": 16
                                                 }}
                                                 """), 
                                                 ("human", "Multiply {num} to itself")])

str_parser = StrOutputParser()
json_parser = JsonOutputParser()

chain1 = prompt1 | llm | str_parser
chain2 = prompt2 | llm | json_parser

transform = RunnableLambda(lambda result: {"num": result})

seq_chain = chain1 | transform | chain2

# seq_chain.invoke({"num1": 2, "num2": 2})


In [7]:
from langchain_community.document_loaders import WebBaseLoader, CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

url = "https://docs.langchain.com/oss/python/langchain/quickstart"
#loader = CSVLoader(file_path="fake_news_data.csv")
loader = WebBaseLoader(url)
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,
    chunk_overlap=50)
chunks = text_splitter.split_documents(raw_docs)

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}
""")

# docs = retriever.invoke("What is LangChain?")

# for i, doc in enumerate(docs):
#     print(f"\n--- Doc {i} ---\n")
#     print(doc.page_content[:500])

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | str_parser
)

rag_chain.invoke("What is langchain?")


USER_AGENT environment variable not set, consider setting it to identify your requests.
/var/folders/gn/mdrgn38s3lzddxysxfdzn12m0000gq/T/ipykernel_4872/1480678092.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

"I don't know"

In [11]:
from langchain_community.chat_models import ChatOllama

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a general knowledge expert. You need to provide name of one country and the output should be only that country name no other text."),
    ("human", "Give me a country name")
])

llm = ChatOllama(model="llama3")

/var/folders/gn/mdrgn38s3lzddxysxfdzn12m0000gq/T/ipykernel_4872/3010654426.py:8: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="llama3")


In [12]:
chain_1 = prompt | llm | str_parser

In [13]:
final_prompt = ChatPromptTemplate.from_template("What is the capital of {country}?")

chain_2 = final_prompt | llm | str_parser

final_chain = chain_1 | {"country": RunnablePassthrough()} | chain_2

In [14]:
chain_1.invoke({})

ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))